# 13 — Pré-processamento Completo: Todos os Questionários Socioeconômicos → Pesos Numéricos

**Objetivo:** Ler os CSVs brutos do ENEM (2012–2024), capturar **todas** as questões disponíveis  
e convertê-las para pesos numéricos — sem carregar os 51M de linhas na RAM do Python.

**Por que ler dos CSVs brutos?**  
Os parquets gerados pelo notebook 02 não incluem 8 questões: Q008 (banheiro), Q009 (quarto),  
Q013 (freezer), Q015 (micro-ondas), Q016 (impressora), Q021 (tel. fixo), Q023, Q025 (trabalha).  
Também não têm SG_UF_PROVA para usar como fallback de região.

**Estratégia:** DuckDB `COPY (...) TO '...' (FORMAT PARQUET)` — cada ano escrito direto no disco,  
Python nunca acumula dados na RAM.

**Entrada:** `datasets/enem/microdados_enem_AAAA/DADOS/MICRODADOS_ENEM_AAAA.csv`  
**Saída:** `data/processed/features_socioeconomicas.parquet`

---

### Cobertura por período

| Questão | Período | Conteúdo |
|---------|---------|----------|
| Q001, Q002 | 2012–2024 | Escolaridade pai / mãe |
| Q006 (harmonizada) | 2012–2024 | Renda familiar em faixas de SM |
| Q003, Q004 | 2019–2024 | Ocupação pai / mãe |
| Q005 | 2019–2024 | Nº de pessoas na residência |
| Q007 | 2019–2024 | Empregada doméstica |
| Q008 **NOVO** | 2019–2024 | Banheiros |
| Q009 **NOVO** | 2019–2024 | Quartos |
| Q010–Q012 | 2019–2024 | Carro, moto, geladeira |
| Q013 **NOVO** | 2019–2023 | Freezer (removido em 2024) |
| Q014 | 2019–2024 | Lavadora (micro-ondas em 2024) |
| Q015 **NOVO** | 2019–2024 | Micro-ondas (lavadora em 2024) |
| Q016 **NOVO** | 2019–2024 | Impressora |
| Q017, Q018–Q020 | 2019–2024 | TV, computador, internet (ver notas 2024) |
| Q021 **NOVO** | 2019–2024 | Telefone fixo (tel. fixo 2023 / posição diferente 2024) |
| Q022 | 2019–2024 | Tipo de escola no EM |
| Q024 | 2019–2023 | Bolsa Família (removida em 2024) |
| Q025 **NOVO** | 2019–2023 | Trabalha (removida em 2024) |

### Notas sobre o questionário 2024
Em 2024 o INEP inseriu uma nova Q006 (renda binária) e removeu Q013/Q024/Q025.  
As posições de lavadora e micro-ondas trocaram. Mapeamentos com baixa confiança são indicados.

## 1. Imports, caminhos e constantes

In [11]:
import duckdb
from pathlib import Path

OUTPUT   = Path('../data/processed/features_socioeconomicas.parquet')
TEMP_DIR = Path('../data/processed/_feat_temp')
TEMP_DIR.mkdir(parents=True, exist_ok=True)

ANOS = list(range(2012, 2025))

def get_csv_path(ano):
    base  = f'../datasets/enem/microdados_enem_{ano}/DADOS/'
    upper = f'{base}MICRODADOS_ENEM_{ano}.csv'
    lower = f'{base}microdados_enem_{ano}.csv'
    return upper if Path(upper).exists() else lower

# ─── Fragmentos SQL reutilizáveis ──────────────────────────────────────────────

def regiao_str(uf_expr):
    return f"""CASE {uf_expr}
        WHEN 'RO' THEN 'Norte'   WHEN 'AC' THEN 'Norte'   WHEN 'AM' THEN 'Norte'
        WHEN 'RR' THEN 'Norte'   WHEN 'PA' THEN 'Norte'   WHEN 'AP' THEN 'Norte'   WHEN 'TO' THEN 'Norte'
        WHEN 'MA' THEN 'Nordeste' WHEN 'PI' THEN 'Nordeste' WHEN 'CE' THEN 'Nordeste'
        WHEN 'RN' THEN 'Nordeste' WHEN 'PB' THEN 'Nordeste' WHEN 'PE' THEN 'Nordeste'
        WHEN 'AL' THEN 'Nordeste' WHEN 'SE' THEN 'Nordeste' WHEN 'BA' THEN 'Nordeste'
        WHEN 'MG' THEN 'Sudeste'  WHEN 'ES' THEN 'Sudeste'  WHEN 'RJ' THEN 'Sudeste'  WHEN 'SP' THEN 'Sudeste'
        WHEN 'PR' THEN 'Sul'      WHEN 'SC' THEN 'Sul'      WHEN 'RS' THEN 'Sul'
        WHEN 'MS' THEN 'Centro-Oeste' WHEN 'MT' THEN 'Centro-Oeste'
        WHEN 'GO' THEN 'Centro-Oeste' WHEN 'DF' THEN 'Centro-Oeste'
    END"""

def regiao_num(uf_expr):
    return f"""CASE {uf_expr}
        WHEN 'RO' THEN 0 WHEN 'AC' THEN 0 WHEN 'AM' THEN 0 WHEN 'RR' THEN 0
        WHEN 'PA' THEN 0 WHEN 'AP' THEN 0 WHEN 'TO' THEN 0
        WHEN 'MA' THEN 1 WHEN 'PI' THEN 1 WHEN 'CE' THEN 1 WHEN 'RN' THEN 1
        WHEN 'PB' THEN 1 WHEN 'PE' THEN 1 WHEN 'AL' THEN 1 WHEN 'SE' THEN 1 WHEN 'BA' THEN 1
        WHEN 'MG' THEN 3 WHEN 'ES' THEN 3 WHEN 'RJ' THEN 3 WHEN 'SP' THEN 3
        WHEN 'PR' THEN 4 WHEN 'SC' THEN 4 WHEN 'RS' THEN 4
        WHEN 'MS' THEN 2 WHEN 'MT' THEN 2 WHEN 'GO' THEN 2 WHEN 'DF' THEN 2
    END"""

def faixa(col):
    return f"""CASE
        WHEN TRY_CAST({col} AS DOUBLE) IS NULL  THEN NULL
        WHEN TRY_CAST({col} AS DOUBLE) <  400   THEN '<400'
        WHEN TRY_CAST({col} AS DOUBLE) <  500   THEN '400-500'
        WHEN TRY_CAST({col} AS DOUBLE) <  600   THEN '500-600'
        WHEN TRY_CAST({col} AS DOUBLE) <  700   THEN '600-700'
        WHEN TRY_CAST({col} AS DOUBLE) <  800   THEN '700-800'
        ELSE '>800'
    END"""

def q_escol(col):
    return f"CASE {col} WHEN 'A' THEN 0 WHEN 'B' THEN 1 WHEN 'C' THEN 2 WHEN 'D' THEN 3 WHEN 'E' THEN 4 WHEN 'F' THEN 5 WHEN 'G' THEN 6 WHEN 'H' THEN 7 END"

def q_renda(col, ate_2014):
    if ate_2014:
        return f"CASE {col} WHEN 'A' THEN 0 WHEN 'B' THEN 1 WHEN 'C' THEN 2 WHEN 'D' THEN 3 WHEN 'E' THEN 3 WHEN 'F' THEN 4 WHEN 'G' THEN 4 WHEN 'H' THEN 4 WHEN 'I' THEN 4 WHEN 'J' THEN 4 END"
    else:
        return f"CASE {col} WHEN 'A' THEN 0 WHEN 'B' THEN 1 WHEN 'C' THEN 2 WHEN 'D' THEN 2 WHEN 'E' THEN 3 WHEN 'F' THEN 3 WHEN 'G' THEN 3 WHEN 'H' THEN 3 WHEN 'I' THEN 4 WHEN 'J' THEN 4 WHEN 'K' THEN 4 WHEN 'L' THEN 4 WHEN 'M' THEN 4 WHEN 'N' THEN 4 WHEN 'O' THEN 4 WHEN 'P' THEN 4 END"

def q_c4(col):
    return f"CASE {col} WHEN 'A' THEN 0 WHEN 'B' THEN 1 WHEN 'C' THEN 2 WHEN 'D' THEN 3 END"

def q_c5(col):
    return f"CASE {col} WHEN 'A' THEN 0 WHEN 'B' THEN 1 WHEN 'C' THEN 2 WHEN 'D' THEN 3 WHEN 'E' THEN 4 END"

def q_c3(col):
    return f"CASE {col} WHEN 'A' THEN 0 WHEN 'B' THEN 1 WHEN 'C' THEN 2 END"

def q_bin(col):
    return f"CASE {col} WHEN 'A' THEN 0 WHEN 'B' THEN 1 END"

def q_internet(col):
    return f"CASE WHEN {col} IS NULL THEN NULL WHEN {col} = 'A' THEN 0 ELSE 1 END"

def q_ocup(col):
    return f"CASE {col} WHEN 'A' THEN 0 WHEN 'B' THEN 1 WHEN 'C' THEN 2 WHEN 'D' THEN 3 WHEN 'E' THEN 4 WHEN 'F' THEN 5 WHEN 'G' THEN 6 WHEN 'H' THEN 7 WHEN 'I' THEN 8 WHEN 'J' THEN 9 END"

def q_npessoas(col, is_int=False):
    if is_int:
        return f"TRY_CAST({col} AS INTEGER)"
    return f"CASE {col} WHEN 'A' THEN 1 WHEN 'B' THEN 2 WHEN 'C' THEN 3 WHEN 'D' THEN 4 WHEN 'E' THEN 5 WHEN 'F' THEN 6 WHEN 'G' THEN 7 END"

# NULLs para 2012-2018 (todas as colunas estendidas)
# IMPORTANTE: trailing comma incluída — não adicionar vírgula nem comentário SQL após {NULLS_EXT}
NULLS_EXT = (
    "NULL AS Q_N_PESSOAS_NUM, NULL AS Q_OCUP_PAI_NUM, NULL AS Q_OCUP_MAE_NUM,\n"
    "        NULL AS Q_EMPREGADA_NUM, NULL AS Q_BANHEIRO_NUM, NULL AS Q_QUARTO_NUM,\n"
    "        NULL AS Q_CARRO_NUM, NULL AS Q_MOTO_NUM, NULL AS Q_GELADEIRA_NUM,\n"
    "        NULL AS Q_FREEZER_NUM, NULL AS Q_LAVADORA_NUM, NULL AS Q_MICROONDAS_NUM,\n"
    "        NULL AS Q_IMPRESSORA_NUM, NULL AS Q_TV_NUM, NULL AS Q_COMPUTADOR_NUM,\n"
    "        NULL AS Q_INTERNET_NUM, NULL AS Q_CELULAR_NUM, NULL AS Q_TEL_FIXO_NUM,\n"
    "        NULL AS Q_TIPO_ESCOLA_EM_NUM, NULL AS Q_BOLSA_FAM_NUM, NULL AS Q_TRABALHA_NUM,"
)

print("Constantes e funções carregadas.")
print(f"Saída: {OUTPUT}")
print(f"Temp:  {TEMP_DIR}")

Constantes e funções carregadas.
Saída: ../data/processed/features_socioeconomicas.parquet
Temp:  ../data/processed/_feat_temp


## 2. SQLs por grupo de anos

### 2012–2018 — apenas colunas base (Q001, Q002, Q006, demográficas, notas)
O questionário estendido (Q007-Q025) existia mas com numeração e semântica diferentes.  
Para comparabilidade temporal, usamos só as questões consistentes.  
REGIAO derivada apenas de SG_UF_ESC (sem SG_UF_PROVA nesses anos).

### 2019–2023 — questionário completo Q001–Q025
REGIAO usa `COALESCE(SG_UF_ESC, SG_UF_PROVA)` → cobre candidatos sem escola registrada.

### 2024 — dois arquivos + remapeamento de questões
- PARTICIPANTES_2024.csv: questionário (Q001-Q023 com nova numeração)
- RESULTADOS_2024.csv: escola, presença e notas
- Join por ROW_NUMBER() (arquivos têm mesma ordenação, sem chave explícita)
- Q006 de 2024 é binária (nova) → usa-se Q007 como renda (equivalente ao Q006 de 2023)
- Questões removidas: freezer (Q013), bolsa família (Q024), trabalha (Q025)
- Micro-ondas e lavadora trocaram de posição

### Confiança dos mapeamentos 2024
| Coluna | CSV 2024 | Confiança | Distribuição esperada vs observada |
|--------|----------|-----------|-----------------------------------|
| banheiro | Q009 | ALTA | B=68% ✓ |
| quarto | Q010 | ALTA | C=52% ✓ |
| micro-ondas | Q014 | ALTA | A=87% ✓ |
| lavadora | Q015 | ALTA | B=61% ✓ |
| impressora | Q016 | ALTA | A=54% ✓ |
| tv | Q018 | ALTA | B=64% ✓ |
| computador | Q019 | ALTA | A=79.5% ✓ (melhor que Q021 usado pelo nb02) |
| internet | Q020 | ALTA | B=88% ✓ |
| tel. fixo | Q017 | BAIXA | A=76% ≈ 2023 Q021 A=79% (tentativo) |
| celular | Q021 | BAIXA | distribuição diferente de 2023 |
| tipo_escola_EM | Q023 | BAIXA | formato alterado em 2024 |

In [12]:
## 3. Processamento por ano → parquets temporários

con = duckdb.connect()
con.execute("SET memory_limit = '4GB'")
con.execute("SET threads = 4")

CSV_OPTS = "delim=';', header=true, encoding='latin-1', ignore_errors=true"
PRESENCA  = "TP_PRESENCA_CN=1 AND TP_PRESENCA_CH=1 AND TP_PRESENCA_LC=1 AND TP_PRESENCA_MT=1 AND NU_NOTA_MT IS NOT NULL"

SCORES_SQL = f"""TRY_CAST(NU_NOTA_CN       AS DOUBLE) AS NU_NOTA_CN,
        TRY_CAST(NU_NOTA_CH       AS DOUBLE) AS NU_NOTA_CH,
        TRY_CAST(NU_NOTA_LC       AS DOUBLE) AS NU_NOTA_LC,
        TRY_CAST(NU_NOTA_MT       AS DOUBLE) AS NU_NOTA_MT,
        TRY_CAST(NU_NOTA_REDACAO  AS DOUBLE) AS NU_NOTA_REDACAO,
        {faixa('NU_NOTA_CN')}                AS FAIXA_CN,
        {faixa('NU_NOTA_CH')}                AS FAIXA_CH,
        {faixa('NU_NOTA_LC')}                AS FAIXA_LC,
        {faixa('NU_NOTA_MT')}                AS FAIXA_MT,
        {faixa('NU_NOTA_REDACAO')}           AS FAIXA_REDACAO"""

# TP_SEXO pode ser INT64 (0/1) em 2012-2014 — cast para VARCHAR antes do CASE.
# Q005 é sempre INTEGER nos raw CSVs (1,2,3...), não letra — usar TRY_CAST diretamente.
def sexo_case(col):
    return f"CASE CAST({col} AS VARCHAR) WHEN 'M' THEN 0 WHEN '0' THEN 0 WHEN 'F' THEN 1 WHEN '1' THEN 1 END AS TP_SEXO_NUM"

for ano in ANOS:
    temp = TEMP_DIR / f'yr_{ano}.parquet'

    # ── 2024: dois arquivos, remapeamento completo de questões ─────────────────
    if ano == 2024:
        base   = '../datasets/enem/microdados_enem_2024/DADOS/'
        p_path = f'{base}PARTICIPANTES_2024.csv'
        r_path = f'{base}RESULTADOS_2024.csv'

        sql = f"""
        COPY (
          WITH
          p AS (
            SELECT ROW_NUMBER() OVER () AS rn,
                   NU_ANO, TP_SEXO, TP_COR_RACA, Q001, Q002,
                   Q007 AS Q006_RAW,
                   Q003, Q004, Q005,
                   Q008 AS _EMPREGADA,
                   Q009 AS _BANHEIRO,
                   Q010 AS _QUARTO,
                   Q011 AS _CARRO,
                   Q012 AS _MOTO,
                   Q013 AS _GELADEIRA,
                   Q014 AS _MICROONDAS,
                   Q015 AS _LAVADORA,
                   Q016 AS _IMPRESSORA,
                   Q017 AS _TEL_FIXO,
                   Q018 AS _TV,
                   Q019 AS _COMPUTADOR,
                   Q020 AS _INTERNET,
                   Q021 AS _CELULAR,
                   Q023 AS _TIPO_ESCOLA
            FROM read_csv('{p_path}', {CSV_OPTS})
          ),
          r AS (
            SELECT ROW_NUMBER() OVER () AS rn,
                   SG_UF_ESC,
                   CASE TP_DEPENDENCIA_ADM_ESC WHEN 4 THEN 3 ELSE 2 END AS TP_ESCOLA_RAW,
                   SG_UF_PROVA,
                   TP_PRESENCA_CN, TP_PRESENCA_CH, TP_PRESENCA_LC, TP_PRESENCA_MT,
                   NU_NOTA_CN, NU_NOTA_CH, NU_NOTA_LC, NU_NOTA_MT, NU_NOTA_REDACAO
            FROM read_csv('{r_path}', {CSV_OPTS})
          )
          SELECT
            2024                                                                    AS NU_ANO,
            r.SG_UF_ESC,
            r.SG_UF_PROVA,
            {regiao_str('COALESCE(r.SG_UF_ESC, r.SG_UF_PROVA)')}                  AS REGIAO,
            {regiao_num('COALESCE(r.SG_UF_ESC, r.SG_UF_PROVA)')}                  AS REGIAO_NUM,
            {sexo_case('p.TP_SEXO')},
            CASE WHEN TRY_CAST(p.TP_COR_RACA AS INTEGER) = 0 THEN NULL ELSE TRY_CAST(p.TP_COR_RACA AS INTEGER) END AS TP_COR_RACA_NUM,
            CASE r.TP_ESCOLA_RAW WHEN 2 THEN 0 WHEN 3 THEN 1 END                  AS TP_ESCOLA_NUM,
            {q_escol('p.Q001')}                                                    AS Q001_NUM,
            {q_escol('p.Q002')}                                                    AS Q002_NUM,
            {q_renda('p.Q006_RAW', False)}                                         AS Q006_NUM,
            TRY_CAST(p.Q005 AS INTEGER)                                            AS Q_N_PESSOAS_NUM,
            {q_ocup('p.Q003')}                                                     AS Q_OCUP_PAI_NUM,
            {q_ocup('p.Q004')}                                                     AS Q_OCUP_MAE_NUM,
            {q_c4('p._EMPREGADA')}                                                 AS Q_EMPREGADA_NUM,
            {q_c5('p._BANHEIRO')}                                                  AS Q_BANHEIRO_NUM,
            {q_c5('p._QUARTO')}                                                    AS Q_QUARTO_NUM,
            {q_c4('p._CARRO')}                                                     AS Q_CARRO_NUM,
            {q_c4('p._MOTO')}                                                      AS Q_MOTO_NUM,
            {q_c4('p._GELADEIRA')}                                                 AS Q_GELADEIRA_NUM,
            NULL                                                                    AS Q_FREEZER_NUM,
            {q_c3('p._LAVADORA')}                                                  AS Q_LAVADORA_NUM,
            {q_bin('p._MICROONDAS')}                                               AS Q_MICROONDAS_NUM,
            {q_bin('p._IMPRESSORA')}                                               AS Q_IMPRESSORA_NUM,
            {q_c4('p._TV')}                                                        AS Q_TV_NUM,
            {q_bin('p._COMPUTADOR')}                                               AS Q_COMPUTADOR_NUM,
            {q_internet('p._INTERNET')}                                            AS Q_INTERNET_NUM,
            {q_c4('p._CELULAR')}                                                   AS Q_CELULAR_NUM,
            {q_bin('p._TEL_FIXO')}                                                 AS Q_TEL_FIXO_NUM,
            {q_c5('p._TIPO_ESCOLA')}                                               AS Q_TIPO_ESCOLA_EM_NUM,
            NULL                                                                    AS Q_BOLSA_FAM_NUM,
            NULL                                                                    AS Q_TRABALHA_NUM,
            {SCORES_SQL}
          FROM p JOIN r ON p.rn = r.rn
          WHERE {PRESENCA}
        ) TO '{temp}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """

    # ── 2019–2023: questionário completo Q001-Q025 ─────────────────────────────
    elif ano >= 2019:
        csv = get_csv_path(ano)
        sql = f"""
        COPY (
          SELECT
            CAST(NU_ANO AS INTEGER)                                                 AS NU_ANO,
            SG_UF_ESC,
            SG_UF_PROVA,
            {regiao_str('COALESCE(SG_UF_ESC, SG_UF_PROVA)')}                       AS REGIAO,
            {regiao_num('COALESCE(SG_UF_ESC, SG_UF_PROVA)')}                       AS REGIAO_NUM,
            {sexo_case('TP_SEXO')},
            CASE WHEN TRY_CAST(TP_COR_RACA AS INTEGER) = 0 THEN NULL ELSE TRY_CAST(TP_COR_RACA AS INTEGER) END AS TP_COR_RACA_NUM,
            CASE TRY_CAST(TP_ESCOLA AS INTEGER) WHEN 2 THEN 0 WHEN 3 THEN 1 END    AS TP_ESCOLA_NUM,
            {q_escol('Q001')}                                                       AS Q001_NUM,
            {q_escol('Q002')}                                                       AS Q002_NUM,
            {q_renda('Q006', False)}                                                AS Q006_NUM,
            TRY_CAST(Q005 AS INTEGER)                                               AS Q_N_PESSOAS_NUM,
            {q_ocup('Q003')}                                                        AS Q_OCUP_PAI_NUM,
            {q_ocup('Q004')}                                                        AS Q_OCUP_MAE_NUM,
            {q_c4('Q007')}                                                          AS Q_EMPREGADA_NUM,
            {q_c5('Q008')}                                                          AS Q_BANHEIRO_NUM,
            {q_c5('Q009')}                                                          AS Q_QUARTO_NUM,
            {q_c4('Q010')}                                                          AS Q_CARRO_NUM,
            {q_c4('Q011')}                                                          AS Q_MOTO_NUM,
            {q_c4('Q012')}                                                          AS Q_GELADEIRA_NUM,
            {q_c3('Q013')}                                                          AS Q_FREEZER_NUM,
            {q_c3('Q014')}                                                          AS Q_LAVADORA_NUM,
            {q_bin('Q015')}                                                         AS Q_MICROONDAS_NUM,
            {q_bin('Q016')}                                                         AS Q_IMPRESSORA_NUM,
            {q_c4('Q017')}                                                          AS Q_TV_NUM,
            {q_c4('Q018')}                                                          AS Q_COMPUTADOR_NUM,
            {q_internet('Q019')}                                                    AS Q_INTERNET_NUM,
            {q_c4('Q020')}                                                          AS Q_CELULAR_NUM,
            {q_bin('Q021')}                                                         AS Q_TEL_FIXO_NUM,
            {q_c5('Q022')}                                                          AS Q_TIPO_ESCOLA_EM_NUM,
            {q_internet('Q024')}                                                    AS Q_BOLSA_FAM_NUM,
            {q_bin('Q025')}                                                         AS Q_TRABALHA_NUM,
            {SCORES_SQL}
          FROM read_csv('{csv}', {CSV_OPTS})
          WHERE {PRESENCA}
        ) TO '{temp}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """

    # ── 2012–2018: apenas base (Q001, Q002, Q006, demográficas, notas) ─────────
    else:
        csv   = get_csv_path(ano)
        ate14 = ano <= 2014
        sql = f"""
        COPY (
          SELECT
            CAST(NU_ANO AS INTEGER)                                                 AS NU_ANO,
            SG_UF_ESC,
            NULL                                                                    AS SG_UF_PROVA,
            {regiao_str('SG_UF_ESC')}                                               AS REGIAO,
            {regiao_num('SG_UF_ESC')}                                               AS REGIAO_NUM,
            {sexo_case('TP_SEXO')},
            CASE WHEN TRY_CAST(TP_COR_RACA AS INTEGER) = 0 THEN NULL ELSE TRY_CAST(TP_COR_RACA AS INTEGER) END AS TP_COR_RACA_NUM,
            CASE TRY_CAST(TP_ESCOLA AS INTEGER) WHEN 2 THEN 0 WHEN 3 THEN 1 END    AS TP_ESCOLA_NUM,
            {q_escol('Q001')}                                                       AS Q001_NUM,
            {q_escol('Q002')}                                                       AS Q002_NUM,
            {q_renda('Q006', ate14)}                                                AS Q006_NUM,
            {NULLS_EXT}
            {SCORES_SQL}
          FROM read_csv('{csv}', {CSV_OPTS})
          WHERE {PRESENCA}
        ) TO '{temp}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """

    print(f"  {ano}...", end=" ", flush=True)
    con.execute(sql)
    size = temp.stat().st_size / 1024**2
    print(f"OK ({size:.0f} MB)")

con.close()
print("\nTodos os anos processados.")

  2012... OK (45 MB)
  2013... OK (55 MB)
  2014... OK (63 MB)
  2015... OK (61 MB)
  2016... OK (63 MB)
  2017... OK (48 MB)
  2018... OK (42 MB)
  2019... OK (56 MB)
  2020... OK (41 MB)
  2021... OK (35 MB)
  2022... OK (36 MB)
  2023... OK (42 MB)
  2024... OK (45 MB)

Todos os anos processados.


## 4. Union dos parquets temporários → arquivo final

In [13]:
import shutil

con = duckdb.connect()
con.execute("SET memory_limit = '4GB'")
con.execute("SET threads = 4")

temp_glob = str(TEMP_DIR / 'yr_*.parquet')
print(f"Unindo parquets de: {temp_glob}")

con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{temp_glob}', union_by_name=true)
        ORDER BY NU_ANO
    )
    TO '{OUTPUT}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.close()

size_mb = OUTPUT.stat().st_size / 1024**2
print(f"Salvo: {OUTPUT}")
print(f"Tamanho: {size_mb:.0f} MB")

# Remove temp
shutil.rmtree(TEMP_DIR)
print("Arquivos temporários removidos.")

Unindo parquets de: ../data/processed/_feat_temp/yr_*.parquet
Salvo: ../data/processed/features_socioeconomicas.parquet
Tamanho: 631 MB
Arquivos temporários removidos.


## 5. Verificação: cobertura e ranges por coluna

In [14]:
con = duckdb.connect()

n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUTPUT}')").fetchone()[0]
print(f"Total de linhas: {n:,}\n")

# Cobertura de todas as colunas numéricas
cols = [
    'REGIAO_NUM','TP_SEXO_NUM','TP_COR_RACA_NUM','TP_ESCOLA_NUM',
    'Q001_NUM','Q002_NUM','Q006_NUM',
    'Q_N_PESSOAS_NUM','Q_OCUP_PAI_NUM','Q_OCUP_MAE_NUM',
    'Q_EMPREGADA_NUM',
    'Q_BANHEIRO_NUM','Q_QUARTO_NUM',
    'Q_CARRO_NUM','Q_MOTO_NUM','Q_GELADEIRA_NUM',
    'Q_FREEZER_NUM','Q_LAVADORA_NUM','Q_MICROONDAS_NUM','Q_IMPRESSORA_NUM',
    'Q_TV_NUM','Q_COMPUTADOR_NUM','Q_INTERNET_NUM','Q_CELULAR_NUM','Q_TEL_FIXO_NUM',
    'Q_TIPO_ESCOLA_EM_NUM','Q_BOLSA_FAM_NUM','Q_TRABALHA_NUM',
]

exprs_pct = ", ".join(f"ROUND(COUNT({c})*100.0/COUNT(*),1) AS pct_{c}" for c in cols)
exprs_min = ", ".join(f"MIN({c}) AS min_{c}" for c in cols)
exprs_max = ", ".join(f"MAX({c}) AS max_{c}" for c in cols)

pct = con.execute(f"SELECT {exprs_pct} FROM read_parquet('{OUTPUT}')").df().T
pct.index = [i.replace('pct_','') for i in pct.index]
pct.columns = ['% nn']

mn = con.execute(f"SELECT {exprs_min} FROM read_parquet('{OUTPUT}')").df().T
mn.index = [i.replace('min_','') for i in mn.index]
mn.columns = ['min']

mx = con.execute(f"SELECT {exprs_max} FROM read_parquet('{OUTPUT}')").df().T
mx.index = [i.replace('max_','') for i in mx.index]
mx.columns = ['max']

print(pct.join(mn).join(mx).to_string())
con.close()

Total de linhas: 51,321,197

                       % nn  min  max
REGIAO_NUM             50.8    0    4
TP_SEXO_NUM           100.0    0    1
TP_COR_RACA_NUM        98.3    1    6
TP_ESCOLA_NUM          28.1    0    1
Q001_NUM               97.3    0    7
Q002_NUM               99.0    0    7
Q006_NUM               99.1    0    4
Q_N_PESSOAS_NUM        32.2    1   20
Q_OCUP_PAI_NUM         32.2    0    5
Q_OCUP_MAE_NUM         32.2    0    5
Q_EMPREGADA_NUM        32.2    0    3
Q_BANHEIRO_NUM         32.2    0    4
Q_QUARTO_NUM           32.2    0    4
Q_CARRO_NUM            32.1    0    3
Q_MOTO_NUM             32.2    0    3
Q_GELADEIRA_NUM        32.2    0    3
Q_FREEZER_NUM          26.2    0    2
Q_LAVADORA_NUM         32.2    0    2
Q_MICROONDAS_NUM       32.1    0    1
Q_IMPRESSORA_NUM       32.0    0    1
Q_TV_NUM               32.2    0    3
Q_COMPUTADOR_NUM       32.2    0    1
Q_INTERNET_NUM         32.2    0    1
Q_CELULAR_NUM          32.1    0    3
Q_TEL_FIXO_NUM       

## 6. Validação por ano (query SQL obrigatória)

In [15]:
con = duckdb.connect()
resultado = con.execute(f"""
    SELECT
        NU_ANO,
        COUNT(*)                               AS candidatos,
        ROUND(COUNT(REGIAO_NUM)*100.0/COUNT(*),1)        AS pct_regiao,
        ROUND(AVG(Q006_NUM), 2)                AS renda_media,
        ROUND(AVG(Q001_NUM), 2)                AS escol_pai,
        ROUND(AVG(Q_N_PESSOAS_NUM), 1)         AS n_pessoas,
        ROUND(AVG(Q_BANHEIRO_NUM), 2)          AS banheiros,
        ROUND(AVG(Q_QUARTO_NUM), 2)            AS quartos,
        ROUND(AVG(Q_MICROONDAS_NUM), 2)        AS microondas,
        ROUND(AVG(Q_IMPRESSORA_NUM), 2)        AS impressora,
        ROUND(AVG(Q_TEL_FIXO_NUM), 2)          AS tel_fixo,
        ROUND(AVG(Q_TRABALHA_NUM), 2)          AS pct_trabalha,
        ROUND(AVG(NU_NOTA_MT), 1)              AS nota_mt
    FROM read_parquet('{OUTPUT}')
    GROUP BY NU_ANO
    ORDER BY NU_ANO
""").df()
print(resultado.to_string(index=False))
con.close()

 NU_ANO  candidatos  pct_regiao  renda_media  escol_pai  n_pessoas  banheiros  quartos  microondas  impressora  tel_fixo  pct_trabalha  nota_mt
   2012     4079886        30.8         0.90       2.65        NaN        NaN      NaN         NaN         NaN       NaN           NaN    509.0
   2013     5007934        27.2         0.89       2.64        NaN        NaN      NaN         NaN         NaN       NaN           NaN    510.5
   2014     5947909        24.5         0.89       2.62        NaN        NaN      NaN         NaN         NaN       NaN           NaN    473.3
   2015     5604905        25.4         2.21       3.12        NaN        NaN      NaN         NaN         NaN       NaN           NaN    468.1
   2016     5818264        26.1         2.12       3.10        NaN        NaN      NaN         NaN         NaN       NaN           NaN    490.2
   2017     4426692        31.3         2.11       3.15        NaN        NaN      NaN         NaN         NaN       NaN           NaN  